# 02: The stock global fits — the machinery

**This notebook targets the development stack (LATW `dev` branch).** It
uses the *installed* stock global fit from the development LISA Analysis
Tools packages (`LISAanalysistools/install.sh`), not the pip releases.
No Colab:

```bash
git clone https://github.com/lisa-analysis-tools/lisa-analysis-tools.git LISAanalysistools
bash LISAanalysistools/install.sh
```

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
os.environ.setdefault("MAKE_DIAGNOSTIC_PLOTS", "0")   # we do not need in-run eryn plots here

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from copy import deepcopy
import dataclasses
from lisatools.utils.constants import *

from lisatools.globalfit.stock import erebor

# Write to our OWN output dir. The stock default (gf_output_gb_no_fg/
# gb_no_fg_test_2) is also what 01 uses, with a different walker/temperature
# shape -- sharing it makes this notebook resume 01's HDF backend and die on a
# log_like shape mismatch. 02 isolates its runs the same way.
import shutil
STOCK_DIR = "./gf_output_stock/"
shutil.rmtree(STOCK_DIR, ignore_errors=True)
erebor.gb_no_fg_lite.general.file_store_dir = STOCK_DIR

# Slightly larger default text so labels/ticks/titles read clearly in the
# rendered figures.
plt.rcParams.update({"font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12,
                     "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11})


cupy not found, using numpy instead. This will be very slow for large runs. Please install cupy and a compatible CUDA version for GPU acceleration.


[`01`](01_GlobalFitQuickstart.ipynb) ran a stock global fit in four
lines and read its products back. This notebook opens the **machinery**
underneath that one-liner: the variant catalogue, the data-in
preprocessing layer, the layered settings the run is built from, the
declarative *recipe* of sampler moves, and — the part you will reach for
when your research needs something new — **how to write your own move
module and drop it into the stack.**

We demo on the lightest variant, **`gb_no_fg_lite`** (a GB-only fit).
[`03`](03_StockGlobalFitGallery.ipynb) is the per-variant gallery;
[`00`](00_SetupAndAtlas.ipynb) places `lisatools.globalfit` in the stack.

### How to read this notebook

Each `##` section opens with a **TL;DR** and one minimal cell; drop into
*"Going deeper"* for the internals. Most cells here are *inspection* (a
stock fit is a cheap Python object until you `build()` it), so they run
instantly — only §5 actually builds and runs a short chain.

## 1. The variant catalogue

**TL;DR.** A *stock global fit* is an installed, versioned run recipe — a
`StockGlobalFit` subclass, not a settings file. The **erebor** family
registers several; `erebor.get_stock_options()` lists them, and each is
reachable as a module attribute (`erebor.gb_no_fg`) or via
`erebor.get_stock("gb_no_fg")`.

In [2]:
for name, desc in erebor.get_stock_options():
    print(f"{name:24s} {desc[:64]}")

all_sources              All six branches (gb, psd, galfor, mbh, emri, sobbh) composed fr
all_sources_lite         Laptop-smoke twin of all_sources: the same six branches and mach
blank                    Minimal single-stage fit on zero data (noise optional via includ
full_year_combined       Catalogue-driven multi-leaf MBH+EMRI+SOBBH fit over the full yea
full_year_combined_lite  Laptop-smoke twin of full_year_combined: one month instead of a 
gb_no_fg                 GB-only fit on the mojito L1 GB galaxy: fixed PSD, no foreground
gb_no_fg_lite            Laptop-smoke twin of gb_no_fg: two-week span, 3 iterations, 4 wa
noise_only               Joint WDM noise fit: instrument PSD + hyperbolic-tangent galacti
noise_only_lite          Laptop-smoke twin of noise_only: quarter-length time grid, 3 ite
noise_sgwb               Joint WDM noise fit: instrument PSD + galactic foreground + powe
noise_sgwb_lite          Laptop-smoke twin of noise_sgwb: quarter-length time grid, 3 ite


Five base variants, each with a `*_lite` laptop-smoke twin:

| variant | what it fits | when to use it |
|---|---|---|
| **`gb_no_fg`** | galactic binaries only, fixed PSD, f > 6 mHz | GB methods; the simplest full run |
| **`all_sources`** | all six branches (gb, psd, galfor, mbh, emri, sobbh) | the real joint global fit |
| **`full_year_combined`** | multi-leaf MBH + EMRI + SOBBH, full year | the heavy-source branches |
| **`noise_only`** | instrument PSD + galactic foreground | noise-model work |
| **`noise_sgwb`** | PSD + foreground + power-law SGWB | stochastic-background work |

`erebor.get_stock(name, **overrides)` and `erebor.<name>(**overrides)` are
equivalent constructors. The per-variant deep demos are in
[`03`](03_StockGlobalFitGallery.ipynb).

In [3]:
fit = erebor.gb_no_fg_lite()      # our demo subject for the rest of the notebook
print(fit.describe())

GBNoForegroundLiteGlobalFit (gb_no_fg_lite) — Laptop-smoke twin of gb_no_fg: two-week span, 3 iterations, 4 walkers x 2 temps, 2 GB repeat proposals, CPU.
  built: False
  headline:
    nwalkers = 4
    ntemps = 2
    dt = 2.5
    num_iterations = 3
    file_store_dir = './gf_output_stock/'
    base_file_name = 'gb_no_fg_test_2'
    gpus = None
    gpu_backend = 'auto'
    tobs_target = 1209600.0
    min_freq = 0.006
    max_freq = 0.025
    tdi_chan = 'XYZ'
    window_tukey_alpha = 0.05
    mojito_data_path = '/Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/'
    use_gpu = False
  branches: ['gb']
    gb: GBNoFgGBSettings
  recipe:
    [pe] gb_pe:
        rj_prior  <- stock branch=gb
  setup_function: setup_recipe


## 2. Data in — the preprocessing layer

**TL;DR.** Every variant turns raw input into the analysis data through a
*data processor*. `lisatools.globalfit.preprocessing` holds the loaders
(`L1DataLoader` for mojito L1, `SangriaDataLoader` for the LDC file) and
the in-process `SyntheticDataProcessor`; `stock/erebor/injections.py` adds
the synthetic per-source stream builders. One knob — `general.data_mode`
(env `DATA_MODE`) — swaps the whole pipeline.

In [4]:
from lisatools.globalfit import preprocessing
loaders = [c for c in dir(preprocessing)
           if c.endswith(("DataLoader", "ProcessingStep", "DataProcessor"))]
print("preprocessing:", loaders)

# the data-mode knob and its default for this variant
print("\ngb_no_fg default data_mode:", fit.general.data_mode)   # "mojito"

preprocessing: ['BaseProcessingStep', 'L1DataLoader', 'L1ProcessingStep', 'SangriaDataLoader', 'SangriaProcessingStep', 'SyntheticDataProcessor', 'SyntheticSourceProcessingStep']

gb_no_fg default data_mode: mojito


### Going deeper: the three data modes and the synthetic fallback

- **`"mojito"`** (default for `gb_no_fg` / `all_sources` /
  `full_year_combined`) — `L1DataLoader` reads a mojito L1 folder
  (per-link time series + source catalogues), builds TDI, and pours it
  onto the analysis grid. For GB it also populates
  `general_info.catalogue["GB"]` so leaves can start at true catalogue
  points (the SNR-cut subset).
- **`"synthetic"`** — no external data: the processor builds each present
  branch's stream *in-process* from injection tables. The tables come
  from `stock/erebor/injections.py` (`make_gb_injections`,
  `make_emri_injections`, `make_mbh_injections`, `make_sobbh_injections`)
  in either `"stock"` (fixed) or `"prior"` (seeded prior draws) mode.
- **`"sangria"`** — `SangriaDataLoader` on the legacy LDC training file
  (`all_sources` only).

**The fallback:** when `data_mode="mojito"` but the mojito folder is
missing, `build()` switches to `"synthetic"` with prior-drawn injections
automatically (`EreborFit.resolve_data_source`), so a bare laptop still
runs. Swap the mode by one assignment:

```python
fit.general.data_mode = "synthetic"   # or DATA_MODE=synthetic
```

An explicit `general.data_processor_class` swap always wins over the
`data_mode` knob — that is the seam for plugging in your own loader.

## 3. The settings blocks + env resolution

**TL;DR.** `fit.general`, the per-branch blocks (`fit.gb`, …) and `fit.recipe`
are **three peers** — sibling attributes of the fit, each swappable on its own.
None contains the others: `fit.general = ...` leaves `fit.gb` and `fit.recipe`
untouched. What genuinely nests is *composition* — the fit holds those three
blocks, and each block is built from **atoms** (transforms, priors,
moves, injection tables) — and the *lifecycle* above it: a
`StockGlobalFit` is the cheap, picklable config; `.build()` turns it into a
**`GlobalFitSetup`** (the heavy built config + state — historically
`CurrentInfoGlobalFit`, still importable as an alias); `GlobalFit(curr).run()`
is the **runner** that drives the sampler.

The one real ordering is **build-time resolution**, not hierarchy: `build()`
resolves `general` first (grid, `Tobs`, domain, data); each branch then
inherits any *unset* `Tobs`/`dt`/`domain_settings`/`log_dir` from it; and the
recipe is materialized last, with every `Move.branch` validated against the
enabled branches. So general's run-wide choices flow *down* into the branches
as defaults — but that is defaulting, not ownership.

Many field defaults are **environment-backed**, resolving *explicit kwarg > env
var > hard default*; a `*_lite` preset slots in just below the env var, giving
*explicit kwarg > env var > lite preset > hard default* (a set env var
overrules the preset).

In [5]:
# the three peer blocks (none owns the others), then the lifecycle above them
print("general      :", type(fit.general).__name__)
print("branches     :", {n: type(getattr(fit, n)).__name__ for n in fit.branches})
print("recipe       :", fit.recipe)
print("lifecycle    :", type(fit).__name__, "-> .build() -> GlobalFitSetup -> GlobalFit(curr).run()")

# EVERY setting at a glance: describe(full=True) dumps all fields of the general
# block AND each branch block (not just the headline knobs); all_settings()
# returns the same content as a nested dict for programmatic use.
print("\n--- fit.describe(full=True): every configured setting ---")
print(fit.describe(full=True))

# env resolution: explicit kwarg > env var > hard default (non-lite fit here)
os.environ["NWALKERS"] = "6"
print("\nenv NWALKERS=6      -> nwalkers =", erebor.gb_no_fg().general.nwalkers)
print("kwarg nwalkers=10   -> nwalkers =", erebor.gb_no_fg(nwalkers=10).general.nwalkers)
del os.environ["NWALKERS"]
print("unset (hard default)-> nwalkers =", erebor.gb_no_fg().general.nwalkers)

general      : GBNoFgGeneralSettings
branches     : {'gb': 'GBNoFgGBSettings'}
recipe       : Recipe(['gb_pe'])
lifecycle    : GBNoForegroundLiteGlobalFit -> .build() -> GlobalFitSetup -> GlobalFit(curr).run()

--- fit.describe(full=True): every configured setting ---
GBNoForegroundLiteGlobalFit (gb_no_fg_lite) — Laptop-smoke twin of gb_no_fg: two-week span, 3 iterations, 4 walkers x 2 temps, 2 GB repeat proposals, CPU.
  built: False
  general (GBNoFgGeneralSettings):
    Tobs = None
    dt = 2.5
    initialize_kwargs = None
    transform = None
    priors = None
    periodic = None
    nleaves_max = None
    nleaves_min = None
    ndim = None
    betas = None
    other_tempering_kwargs = None
    branch_state = None
    branch_backend = None
    log_dir = None
    signal_gen = None
    num_iterations = 3
    file_store_dir = './gf_output_stock/'
    base_file_name = 'gb_no_fg_test_2'
    main_file_key = 'testing'
    past_file_for_start = None
    orbits = None
    gpu_orbits = None


### Going deeper: `env_default` / `env_resolve`

**Naming rule.** An env knob is always the **capitalised attribute name** of
the field it seeds — `general.data_mode` → `DATA_MODE`,
`general.num_iterations` → `NUM_ITERATIONS` — so the knob is derivable from the
attribute and vice versa, with nothing to look up. Per-branch blocks prefix the
branch namespace (`gb.min_freq` → `GB_MIN_FREQ`), because the bare name would
collide with the general block's field of the same name.

The env-backed dataclass fields use
`lisatools.globalfit.stock.base.env_default(VAR, default, cast)` as their
`default_factory`: the factory reads the environment **at construction time**,
so `NUM_ITERATIONS`, `NWALKERS`, `NTEMPS`, `TOBS_TARGET`, `DATA_MODE`,
`USE_GPU`/`GPUS`/`GPU_BACKEND`, `FILE_STORE_DIR` all just seed the corresponding
knob's default. Because they are *only* defaults, an explicit constructor kwarg
or a later `fit.general.x = ...` assignment always wins. The `*_lite` twins pin
a fixed smoke-size preset that sits **below** any set env var: a companion
`lite_env_vars()` map means that whenever a preset knob's env var is set
(`NWALKERS=16`, `USE_GPU=1`, …) the env value is kept and the preset is skipped
for that knob — so the full precedence is *explicit kwarg > env var > lite
preset > hard default* (see
[`01` §Env knobs and precedence](01_GlobalFitQuickstart.ipynb)).

**Older spellings.** A few knobs were renamed to follow the rule
(`GF_NUM_ITER` → `NUM_ITERATIONS`, `DATA_PROCESSOR` → `DATA_MODE`,
`MAKE_PLOTS` → `MAKE_DIAGNOSTIC_PLOTS`, `WAVELET_DUR_*` →
`WAVELET_DURATION_*`). The old names still work but raise a
`DeprecationWarning`: an unrecognised env var is *silently ignored*, so a hard
rename would have quietly downgraded existing runbooks rather than failing
loudly. `base.ENV_ALIASES` is the canonical → legacy table; prefer the
canonical name in new code.

### Reaching (almost) every knob

There is no settings file to edit and no registry to consult: every knob a
stock fit holds is plain attribute access on the three peer blocks. Six paths
cover the whole surface.

| reach | how |
|---|---|
| the headline knobs | `fit.nwalkers = 8` — delegates to `fit.general` |
| any **general** field | `fit.general.<field> = ...` |
| any **branch** field | `fit.<branch>.<field> = ...` (`fit.gb`, `fit.mbh`, …) |
| the **recipe** | `fit.recipe.add_move(...)` / `.pop_move(...)` / `.set_move_debug(...)` |
| a whole **block** | `fit.general = ...`, `fit.add_branch(...)`, `fit.remove_branch(...)` |
| at construction / the shell | `erebor.gb_no_fg(nwalkers=8)` · `NWALKERS=8 python ...` |

To *find* a knob rather than guess it: `fit.describe(full=True)` prints every
field of every block, and `fit.all_settings()` hands you the same thing as a
nested dict you can iterate. The "(almost)" is the honest part — a handful of
fields are left `None` on purpose and **derived at build time** (`Tobs` from the
grid, `domain_settings` from the band knobs, `window_taper_duration` from
`window_tukey_alpha`). Setting those explicitly overrides the derivation, so
swap whole objects rather than fighting the resolver.

In [6]:
fit9 = erebor.gb_no_fg_lite()

# 1. a headline knob -- shorthand that delegates to the general block
fit9.nwalkers = 8
print("headline  fit9.nwalkers = 8            -> fit9.general.nwalkers =", fit9.general.nwalkers)

# 2. any general field, by its own name
fit9.general.random_seed = 42
print("general   fit9.general.random_seed     ->", fit9.general.random_seed)

# 3. any branch field, by its own name
fit9.gb.num_repeat_proposals = 7
print("branch    fit9.gb.num_repeat_proposals ->", fit9.gb.num_repeat_proposals)

# 4. the recipe (the move stack itself is a knob)
fit9.recipe.set_move_debug("rj_prior", False)
print("recipe    fit9.recipe.move_names()     ->", fit9.recipe.move_names())

# 5. a whole block -- and note the peers: swapping general leaves gb/recipe alone
gb_before, recipe_before = fit9.gb, fit9.recipe
fit9.general = deepcopy(fit9.general)
print("block     swapped fit9.general         -> gb untouched:", fit9.gb is gb_before,
      "| recipe untouched:", fit9.recipe is recipe_before)

# 6. at construction (kwarg) -- and from the shell via the capitalised name
print("kwarg     erebor.gb_no_fg_lite(nwalkers=3) ->",
      erebor.gb_no_fg_lite(nwalkers=3).general.nwalkers, " (shell: NWALKERS=3 ...)")

# ...and how to ENUMERATE the whole surface rather than guess at it
s = fit9.all_settings()
n_branch = sum(len(v) for v in s["branches"].values())
print(f"\nevery knob: {len(s['general'])} general fields + {n_branch} branch fields "
      f"across {list(s['branches'])} + {len(s['recipe'])} recipe stage(s)")
print("first few general field names:", [f.name for f in dataclasses.fields(fit9.general)][:6], "...")
print("\nfit.describe(full=True) prints all of them; fit.all_settings() returns them as a dict.")

headline  fit9.nwalkers = 8            -> fit9.general.nwalkers = 8
general   fit9.general.random_seed     -> 42
branch    fit9.gb.num_repeat_proposals -> 7
recipe    fit9.recipe.move_names()     -> ['rj_prior']
block     swapped fit9.general         -> gb untouched: True | recipe untouched: True
kwarg     erebor.gb_no_fg_lite(nwalkers=3) -> 3  (shell: NWALKERS=3 ...)

every knob: 80 general fields + 65 branch fields across ['gb'] + 1 recipe stage(s)
first few general field names: ['Tobs', 'dt', 'initialize_kwargs', 'transform', 'priors', 'periodic'] ...

fit.describe(full=True) prints all of them; fit.all_settings() returns them as a dict.


### The knobs worth knowing

`all_sources` — the six-branch flagship — exposes **287 fields** (92 general +
195 across `gb`/`psd`/`galfor`/`mbh`/`emri`/`sobbh`). Almost all of them have a
sensible default you will never touch. These are the ones you actually turn.
Every env name is just the capitalised attribute, so the middle column is
derivable, not memorised.

**Run shape** — how much sampling you do

| knob | env | what it does |
|---|---|---|
| `general.num_iterations` | `NUM_ITERATIONS` | sampler iterations to run |
| `general.nwalkers` | `NWALKERS` | ensemble walkers per temperature |
| `general.ntemps` | `NTEMPS` | tempering rungs; rung 0 is the cold chain you keep |
| `general.random_seed` | — | seeds the run |

**Grid & band** — the analysis domain everything is poured onto

| knob | env | what it does |
|---|---|---|
| `general.dt` | — | data cadence in seconds |
| `general.tobs_target` | `TOBS_TARGET` | target span; the WDM grid is derived to bracket it |
| `general.nf`, `general.nt` | — | fixed-grid override — set **both** and `Tobs = nf·nt·dt` exactly |
| `general.min_freq`, `general.max_freq` | — | the analysis band (Hz) |
| `general.wavelet_duration_min/max` | `WAVELET_DURATION_MIN/MAX` | bounds the derived WDM layer duration |
| `general.window_tukey_alpha` | `WINDOW_TUKEY_ALPHA` | Tukey taper fraction; the WDM edge crop follows from it |

**Data** — where the streams come from

| knob | env | what it does |
|---|---|---|
| `general.data_mode` | `DATA_MODE` | `mojito` (L1 files) · `synthetic` (in-process) · `sangria` (legacy) |
| `general.synthetic_injections` | `SYNTHETIC_INJECTIONS` | `stock` fixed tables · `prior` seeded draws |
| `general.synthetic_injection_seed` | `SYNTHETIC_INJECTION_SEED` | makes those prior draws reproducible |
| `general.mojito_data_path` | `MOJITO_DATA_PATH` | where the L1 tree lives; if absent, `build()` falls back to synthetic |
| `general.gb_injection_params` | — | explicit GB truth rows the synthetic stream is built from |

**Noise** — what the likelihood divides by

| knob | env | what it does |
|---|---|---|
| `general.psd_from_noise_file` | `PSD_FROM_NOISE_FILE` | fit `Soms_d`/`Sa_a` off the mojito NOISE brick instead of stock levels |
| `general.noise_file` | `NOISE_FILE` | that brick's path; `None` auto-discovers it |
| `general.fixed_psd_params` | — | `[Soms_d, Sa_a]` used when no `psd` branch is being fit |

**Compute** — chosen at construction, never per call

| knob | env | what it does |
|---|---|---|
| `general.use_gpu` | `USE_GPU` | force CPU/GPU; unset auto-detects |
| `general.gpus` | `GPUS` | device indices, e.g. `"0,1"` |
| `general.gpu_backend` | `GPU_BACKEND` | which CUDA wheel flavour (`cuda11x`/`12x`/`13x`) |

**Output**

| knob | env | what it does |
|---|---|---|
| `general.file_store_dir` | `FILE_STORE_DIR` | the run directory |
| `general.base_file_name` | `BASE_FILE_NAME` | HDF5 stem — **distinct per run**, or the backend resumes the old one |
| `general.make_diagnostic_plots` | `MAKE_DIAGNOSTIC_PLOTS` | in-run eryn plots off/on |
| `general.plot_iterations` | `PLOT_ITERATIONS` | iterations between plot refreshes |

**Per branch** — `gb` shown; every branch block carries its own equivalents

| knob | env | what it does |
|---|---|---|
| `gb.min_freq`, `gb.max_freq` | `GB_MIN_FREQ`, `GB_MAX_FREQ` | the branch's band, snapped inward to WDM layers |
| `gb.center_freq`, `gb.n_layers` | `GB_CENTER_FREQ`, `GB_N_LAYERS` | the same band expressed as centre + layer count (≥ 3) |
| `gb.nleaves_max`, `gb.nleaves_min` | `GB_NLEAVES_MAX` | how many sources the branch may hold — the RJ range |
| `gb.num_repeat_proposals` | `GB_NUM_REPEAT_PROPOSALS` | in-model proposals per iteration (the main runtime dial) |
| `gb.mode` | `GB_MODE` | `pe` (leaves start at truth) · `search` (start from zero leaves) |
| `gb.use_chirp_mass` | `GB_USE_CHIRP_MASS` | sample `Mc` instead of `fdot` |
| `gb.transform`, `gb.priors`, `gb.ndim` | — | the branch's parameter basis — the atoms inside the block |
| `gb.injection` | — | the branch's truth table |

**`all_sources` only**

| knob | env | what it does |
|---|---|---|
| `general.mojito_source_ids` | — | which catalogue sources per class — and, in synthetic mode, **how many** |
| `general.fit_sgwb` | `FIT_SGWB` | add a power-law SGWB branch to the noise fit |
| `general.add_instrument_noise` | — | `True` auto-resolves to the real NOISE brick, else a synthetic draw |

Everything else is reachable the same way and printed by
`fit.describe(full=True)` — the table above is a starting point, not the
boundary.

### One pattern, extended per branch

A branch is not just its `*Settings` block. Four base classes are extended for
each source class, and the split tells you something real:

| base | defined in | what the branch's version adds |
|---|---|---|
| `Settings` | `globalfit/engine.py` | the branch's knobs |
| `Setup` | `globalfit/engine.py` | the built twin — same fields, resolved |
| eryn `State` | `globalfit/state.py` | its per-source bookkeeping |
| eryn `HDFBackend` | `globalfit/hdfbackend.py` | persistence for exactly that |

The four **source** branches have all four; the **noise** branches (`psd`,
`galfor`, `sgwb`) stop at `Settings`/`Setup`. That is not an omission — a
source branch carries per-source tempering that has to survive a restart (GB a
per-**band** ladder in `GBState.band_info`; mbh/emri/sobbh a per-**leaf** ladder
in `betas_all`), so each needs a State to hold it and a backend to persist it.
The noise branches have no such bookkeeping, so they need neither. Each backend
stores precisely what its State holds.

In [7]:
from lisatools.globalfit.engine import Settings, Setup
from eryn.state import State as ErynState
from eryn.backends import HDFBackend as ErynBackend

QUARTET = [
    ("gb",     erebor.GBSettings,     erebor.GBSetup,     erebor.GBState,    erebor.GBHDFBackend),
    ("mbh",    erebor.MBHSettings,    erebor.MBHSetup,    erebor.MBHState,   erebor.MBHHDFBackend),
    ("emri",   erebor.EMRISettings,   erebor.EMRISetup,   erebor.EMRIState,  erebor.EMRIHDFBackend),
    ("sobbh",  erebor.SOBBHSettings,  erebor.SOBBHSetup,  erebor.SOBBHState, erebor.SOBBHHDFBackend),
    ("psd",    erebor.PSDSettings,    erebor.PSDSetup,    None,              None),
    ("galfor", erebor.GalForSettings, erebor.GalForSetup, None,              None),
]
_n = lambda c: c.__name__ if c is not None else "—"
print(f"{'branch':7s} {'Settings':22s} {'Setup':12s} {'State':10s} HDFBackend")
for b, se, su, st, bk in QUARTET:
    print(f"{b:7s} {_n(se):22s} {_n(su):12s} {_n(st):10s} {_n(bk)}")

# the four bases really are the four bases
print("\nGBSettings -> Settings   :", issubclass(erebor.GBSettings, Settings))
print("GBSetup    -> Setup      :", issubclass(erebor.GBSetup, Setup))
print("GBState    -> eryn State :", issubclass(erebor.GBState, ErynState))
print("GBHDFBackend -> eryn HDFBackend:", issubclass(erebor.GBHDFBackend, ErynBackend))

# NOTE: the State/Backend are wired ONTO the branch block at BUILD (the Setup's
# init_state_backend_info), so fit.gb.branch_state is still None out here.
print("\nbefore build, fit.gb.branch_state =", erebor.all_sources().gb.branch_state)

branch  Settings               Setup        State      HDFBackend
gb      GBSettings             GBSetup      GBState    GBHDFBackend
mbh     MBHSettings            MBHSetup     MBHState   MBHHDFBackend
emri    EMRISettings           EMRISetup    EMRIState  EMRIHDFBackend
sobbh   SOBBHSettings          SOBBHSetup   SOBBHState SOBBHHDFBackend
psd     PSDSettings            PSDSetup     —          —
galfor  GalForSettings         GalForSetup  —          —

GBSettings -> Settings   : True
GBSetup    -> Setup      : True
GBState    -> eryn State : True
GBHDFBackend -> eryn HDFBackend: True

before build, fit.gb.branch_state = None


## 4. Recipes — stages and moves

**TL;DR.** `fit.recipe` is a `Recipe`: an ordered list of `Stage` blocks,
each holding `Move` objects. A move IS a proposal — one concept, one
`add_move` entrance. Everything is cheap and picklable before the run; at
run start every move's required `setup(ctx)` hook runs (heavy construction
happens there) and the recipe materializes itself into the runtime steps
that drive the sampler. Edit the stack with `add_move` / `pop_move` /
`add_stage` / `pop_stage` / `set_move_debug`, all before `build()` — or even
mid-run (`add_move` during `fit.sample()` starts firing on the next
iteration).


In [8]:
demo = erebor.gb_no_fg_lite()
print("before:")
print(demo.list_moves())

# add a stock move to the gb_pe stage (an f-statistic MCMC refinement move),
# then take it back out -- pure declarative editing, no build. A stock name
# is enough; add_move also takes a plain fn(model, state), a Move subclass,
# or a constructed eryn move.
demo.add_move("rj_fstat_mcmc", branch="gb", stage="gb_pe")
print("\nafter add_move:")
print(demo.list_moves())

demo.pop_move("rj_fstat_mcmc")
print("\nafter pop_move:", demo.recipe.move_names())


before:
[pe] gb_pe:
    rj_prior  <- stock branch=gb

after add_move:
[pe] gb_pe:
    rj_prior  <- stock branch=gb
    rj_fstat_mcmc  <- stock branch=gb

after pop_move: ['rj_prior']


### Going deeper: `Move`, `Stage`, `Recipe.setup`

- **`Move(name, branch=None, debug=None)`** — one move (= one proposal),
  with a required **`setup(ctx)`** hook run at materialization. The base
  class's `setup` is the **stock lookup**: the variant's `setup_function`
  supplies a move under this `name` (e.g. `"rj_prior"`, `"psd_pe"`,
  `"mbh_pe"`). To customize, either **subclass `Move` and override
  `setup(ctx)`** (build and return your eryn move from the live context —
  the plug-in seam, §5), or hand `add_move` a plain
  `fn(model, state) -> (new_state, accepted)` function (wrapped in a
  `FunctionMove`). A constructed move object also works but may break
  pickling.
- **`Stage(name, kind, moves, step_kwargs, combine_kwargs)`** — one recipe
  stage; `kind` is `"search"` / `"pe"` / `"rj"`, selecting the RecipeStep
  class its own `setup(ctx)` produces.
- **`Recipe.setup(ctx)`** runs at build time with
  `ctx = MoveBuildContext(recipe, engine_info, curr, acs, priors, state,
  stock_moves, ntemps, nwalkers)`: it calls every move's `setup(ctx)`,
  wraps each stage's moves in a `GFCombineMove`, and registers the runtime
  steps on the same object — `fit.recipe` IS what runs. The stock move
  builders (`build_gb_moves`, `build_psd_moves`, …) construct exactly the
  names the recipe asks for (`recipe.stock_names()`).

Full reference:
`LISAanalysistools/docs/stock-stages-and-moves.md`.


### Why stages exist: changing the setup *mid-run*

A stage is a **plan step**, not just a grouping. At run time the `Recipe` is
handed to the sampler as its **stopping function** and consulted every
iteration (`run.py`: `stopping_fn=self.recipe, stopping_iterations=1`). Each
step answers two questions:

| method | question |
|---|---|
| `stopping_function(i, sample, sampler)` | am I done? |
| `setup_run(i, sample, sampler)` | how should the sampler be set up for me? |

When a step reports done, the `Recipe` records it **in the HDF backend**
(`completed_recipe_step` — so a resume picks up mid-recipe) and calls the next
step's `setup_run`, which does `sampler.moves = step.moves`: it reconfigures
the **live** sampler. The state carries straight over.

The three stock kinds differ only in *where the criterion lives*:

| `kind` | step class | done when |
|---|---|---|
| `"search"` | `SearchRecipeStep` | the search completes — the criterion is handled **internally**, inside the move |
| `"rj"` | `RJRecipeStep` | the cold-chain leaf count **plateaus** (a recipe-level criterion) |
| `"pe"` | `PERecipeStep` | never on its own — it runs on |

**What that buys you: search → PE without leaving the run.** Declare the search
stage before the PE stage and the hand-off is automatic — the leaves the search
found are already in the state. Do it outside and you would run a search, save
the HDF5, reconfigure by hand, reload and restart for PE: four hand-offs, each
a chance to mis-seed the state, none of it recorded in the run.

In [9]:
from lisatools.globalfit import Move, Stage

# A two-stage plan: search first, then PE -- one run, no manual hand-off.
plan = erebor.gb_no_fg_lite()
plan.recipe.add_stage(
    Stage(
        "gb_search",
        kind="rj",                                   # -> RJRecipeStep
        moves=[Move("rj_prior_search", branch="gb")],
        step_kwargs=dict(convergence_iter=5, plateau_branch="gb"),
    ),
    before="gb_pe",
)
print(plan.list_moves())
print("stage kinds ->", [(s.name, s.kind) for s in plan.recipe.stages])

# At run time: stage 1 samples until its leaf count plateaus, the Recipe marks it
# complete in the backend, then calls stage 2's setup_run() -> sampler.moves is
# swapped in place and PE continues from the leaves the search just found.

[rj] gb_search:
    rj_prior_search  <- stock branch=gb
[pe] gb_pe:
    rj_prior  <- stock branch=gb
stage kinds -> [('gb_search', 'rj'), ('gb_pe', 'pe')]


## 5. Add your own source class

**TL;DR.** Everything above was *configuration*. This section adds a **new
source class** — a branch the stock code has never heard of.

Call it the **`sinusoid`**. Pretend it is something entirely different from a
galactic binary: its own branch, state, settings, prior and move. (Underneath
it happens to call the GB time-domain generator, because a galactic binary
*is* a slowly-chirping sinusoid — the waveform is borrowed, the machinery is
all ours. Nothing below touches the `gb` branch.)

A source class is five pieces:

| piece | what it is | why the engine needs it |
|---|---|---|
| `State` | per-branch bookkeeping | carries the per-leaf temperature ladder the move reads |
| `Settings` / `Setup` | the knob block | `ndim`, `nleaves_*`, prior, transform, `injection` |
| waveform | params → `DomainBase` | the actual physics |
| `signal_gen` | sampling params → template | so the **engine** can subtract your source from the residual |
| move | `propose(model, state)` | how the branch is sampled |

A module is still **any object with `propose(model, state) -> (new_state,
accepted)`** — that contract is the plug-in point of the whole global fit, and
it need not be an MCMC proposal at all. But you rarely write one from scratch:
`SingleSourcePEBuilder` is the shared core MBH, EMRI and SOBBH all use, and
`SOBBHMoveBuilder` is *only* `branch_name = "sobbh"` over it. Ours is the same
trick.

### 5.0 The simple interface first: a plain function move

Before the full source-class machinery below, know that **most custom modules
need none of it**. A module is any move — and a move is any plain function
with the eryn proposal signature. You store your own information in your own
closure/object; the only shared surface you touch is
`model.analysis_container_arr` (read the residual, adjust it, write it back):

- `fit.add_move(my_fn)` wraps the function in a `FunctionMove` and handles the
  bookkeeping under the hood (acceptance normalization + re-syncing
  `state.log_like` from the residual so the saved chain stays consistent);
- `fit.add_branch(name, ndim=..., priors=..., moves=[my_fn])` appends a branch
  as **plain branch info** — no `Settings`/`Setup` classes (those belong to
  pre-installed stock code) — wrapped in the basic single-stage recipe. Every
  info-appended branch must be targeted by at least one move;
- `fit.sample()` is the generator run mode: yield per iteration, mutate the
  residual/state in place inside the loop, and the next iteration continues
  from them. `erebor.blank` is the zero-branch canvas made for exactly
  this — all-zero data by default, so the null log-like is exactly 0
  (`include_noise=True` adds a synthetic noise realization).

The two entrances, on the blank fit:


In [10]:
import shutil as _shutil

from eryn.prior import uniform_dist

_shutil.rmtree("./gf_output_blank/", ignore_errors=True)

# Entrance A -- add your branch + move, then run generally:
simple = erebor.blank(nwalkers=4, ntemps=2, num_iterations=3,
                      make_diagnostic_plots=False)

def my_move(model, state):
    aca = model.analysis_container_arr        # the per-walker residuals
    return state, None                        # or (new_state, accepted)

simple.add_branch("line", ndim=2,
                  priors={0: uniform_dist(1e-3, 8e-3), 1: uniform_dist(0.0, 1.0)},
                  moves=[my_move])
print(simple.list_moves())

# Entrance B -- the generator: adjust things inside the loop yourself.
# (They compose: the branch + move above run inside the generator too, and a
# mid-loop simple.add_move(...) starts firing on the next iteration.)
# Storage is a choice: the added branch is recorded to the HDF backend by the
# default storage machinery every step; pass store=False to skip that and
# record what you want yourself inside the loop.
for model, state in simple.sample(iterations=3, progress=False):
    print("cold-chain log-like:", np.round(state.log_like[0], 2))


[pe] main:
    my_move  <- FunctionMove branch=line
2026-07-16 23:25:13,830 - GeneralSetup - DEBUG - Saving h5 backend to ./gf_output_blank/blank_fit_testing.h5


2026-07-16 23:25:13,831 - GeneralSetup - DEBUG - Saving artifacts to ./gf_output_blank/blank_fit_artifacts/


2026-07-16 23:25:13,835 - GeneralSetup - INFO - Using fixed PSD kwargs: {'psd_params': [1.5e-11, 3e-15], 'galfor_params': None}


2026-07-16 23:25:13,835 - GeneralSetup - DEBUG - Preprocess setting: plot_folder = ./gf_output_blank/blank_fit_artifacts/


2026-07-16 23:25:13,836 - GeneralSetup - DEBUG - Preprocess setting: highpass_kwargs = None


...

2026-07-16 23:25:14,443 - lisatools.globalfit.run - INFO - rebuilt residuals from state coords/inds.


2026-07-16 23:25:14,444 - GlobalFit - DEBUG - acs setup done


2026-07-16 23:25:14,450 - lisatools.globalfit.run - INFO - initial log likelihood: [-0. -0. -0. -0.]


need to setup moves that use parallel resources
2026-07-16 23:25:14,472 - lisatools.globalfit.recipe - DEBUG - Setting periodicity of move <lisatools.globalfit.moves.globalfitmove.GFCombineMove object at 0x148e12a50> to <eryn.utils.periodic.PeriodicContainer object at 0x142708fe0>


cold-chain log-like: [-0. -0. -0. -0.]
cold-chain log-like: [-0. -0. -0. -0.]
cold-chain log-like: [-0. -0. -0. -0.]


When is that not enough? When your module is a genuine **source class** —
its template must live in the shared residual (so the engine can subtract it),
it needs its own per-branch state/backend extensions, priors/transforms in a
reusable knob block, and a move built from the live run objects. That is the
rest of this section.


### 5.1 The state and the knob block

`GBState` carries a per-**band** ladder (`band_info`); MBH/EMRI/SOBBH carry a
per-**leaf** ladder (`betas_all`). We want the add/remove move, which reads the
per-leaf kind — so the sinusoid brings a SOBBH-shaped state of its own. This is
the quartet from §3 (`Settings` / `Setup` / `State` / `HDFBackend`) showing up
for real.

The `Settings` base already carries `transform` / `priors` / `periodic` /
`ndim` / `nleaves_*` / `betas` / `branch_state` / `branch_backend` /
`signal_gen`. `SingleSourcePEBuilder.build` reads exactly four fields off the
Setup — `betas`, `ndim`, `nleaves_max`, `waveform_kwargs` — so
`waveform_kwargs` is the only extra a new source class strictly must declare.
We add `injection` (our truth) and `num_prop_repeats` as well.

In [11]:
import typing
import gbgpu                      # registers the gbgpu_<flavor> backend the TD generator needs
from eryn.state import State as ErynState
from lisatools.globalfit.engine import Settings, Setup

BRANCH = "sinusoid"


class SinusoidState(ErynState):
    """Per-leaf temperature ladder -- the SOBBH-shaped state.

    The add/remove move reads ``betas_all`` (one row per leaf). GBState's
    per-band ladder cannot serve it, which is why our source class brings its
    own state instead of borrowing the gb branch's.
    """

    remove_kwargs = ["betas_all"]

    def __init__(self, possible_state, betas_all=None, copy=False, **kwargs):
        if isinstance(possible_state, self.__class__):
            dc = deepcopy if copy else (lambda x: x)
            self.betas_all = dc(possible_state.betas_all)
            self.num_sinusoids = betas_all.shape[0] if betas_all is not None else 1
        else:
            self.betas_all = betas_all
            self.num_sinusoids = possible_state[BRANCH].shape[-2]

    @property
    def reset_kwargs(self):
        return dict(num_sinusoids=self.num_sinusoids)


@dataclasses.dataclass
class SinusoidSettings(Settings):
    """Our knob block: the base carries transform/priors/ndim/nleaves_*/...,
    these are the extras this source class needs."""

    injection: typing.Optional[np.ndarray] = None    # SAMPLING basis (nleaves_max, ndim)
    num_prop_repeats: int = 1
    waveform_kwargs: typing.Optional[dict] = None
    inner_moves: typing.Optional[list] = None


class SinusoidSetup(Setup, SinusoidSettings):
    """Built twin of SinusoidSettings -- mirrors GBSetup(Setup, GBSettings)."""


print("defined:", SinusoidState.__name__, "+", SinusoidSetup.__name__)

defined: SinusoidState + SinusoidSetup


### 5.2 The waveform, and the seam that puts it in the residual

Two objects that are easy to conflate:

- the **waveform** is what the *move* calls to score a proposal;
- the **`signal_gen`** is what the *engine* calls, once per live leaf, to build
  your template and **subtract it from every walker's residual**
  (`run.py::setup_acs`).

A branch with no `signal_gen` is skipped: its signal stays in the residual
forever and the log-like never returns to ~0 at truth, however good the move
is. `signal_gen` takes **sampling** params and applies the transform itself —
the contract stock's `SourceSignalGen` implements.

It needs the runtime `general_info`, which only exists after `build()`, so it
is attached afterwards. The heavy generator lives in a module-level cache,
never on the settings — the pre-build config must stay picklable.

In [12]:
from lisatools.domains import TDSignal
from lisatools.response.tdionfly import GBTDIonTheFly
from lisatools.response.tdiconfig import TDIConfig
from lisatools.globalfit.stock.erebor.transforms import make_gb_transform_container

BACKEND = "cpu"


class SinusoidTDWave:
    """Our source class's waveform: a TD sinusoid, via the GB TD generator.

    Takes the WAVEFORM basis (amp, f0, fdot, fddot, phi0, inc, psi, lam, beta)
    -- exactly what make_gb_transform_container() emits, in that order, so it is
    a bare splat -- and projects onto the run's domain.
    """

    def __init__(self, gi, n_nodes=4096):
        grid = np.asarray(gi.data_td_settings.t_arr)
        t_ref = getattr(gi.data_td_settings, "t0", float(grid[0]))
        self.gen = GBTDIonTheFly(
            np.linspace(float(grid[0]), float(grid[-1]), n_nodes),
            float(gi.Tobs), float(t_ref), 1.0 / float(gi.dt), 1,
            tdi_config=TDIConfig("2nd generation", force_backend=BACKEND),
            orbits=gi.orbits, tdi_chan="XYZ", force_backend=BACKEND,
        )
        self.grid, self.td_settings = grid, gi.data_td_settings
        self.target_domain, self.nchannels = gi.domain_settings, gi.nchannels

    def __call__(self, *params, **kwargs):
        p = np.asarray(params, dtype=float).reshape(9, 1)
        out = self.gen(*p, convert_to_ra_dec=False, return_spline=True)
        B = np.asarray(out.eval_tdi(self.grid))
        B = (B[0] if B.ndim == 3 else B)[: self.nchannels]
        return TDSignal(B, self.td_settings).transform(self.target_domain)


_WAVE_CACHE = {}


def get_sinusoid_wave(gi):
    """One generator per run, cached here: heavy and unpicklable, so it never
    lives on the settings."""
    if id(gi) not in _WAVE_CACHE:
        _WAVE_CACHE[id(gi)] = SinusoidTDWave(gi)
    return _WAVE_CACHE[id(gi)]


class SinusoidSignalGen:
    """The engine seam: sampling params -> template, subtracted by setup_acs."""

    def __init__(self, transform, general_info):
        self.transform, self.general_info = transform, general_info

    def __call__(self, *params, **kwargs):
        params_in = self.transform.both_transforms(np.asarray(params, dtype=float))
        return get_sinusoid_wave(self.general_info)(*params_in, **kwargs)


print("defined: SinusoidTDWave + SinusoidSignalGen")

defined: SinusoidTDWave + SinusoidSignalGen


In [13]:
from eryn.moves import StretchMove
from lisatools.globalfit import Move
from lisatools.globalfit.recipe import SingleSourcePEBuilder


class SinusoidMoveBuilder(SingleSourcePEBuilder):
    """The sinusoid twin of SOBBHMoveBuilder: same shared core, our branch."""

    branch_name = BRANCH


class SinusoidMove(Move):
    """Move with a setup hook: build the wave wrap + shared PE move at run start.

    Nothing is seeded here -- the engine already brought our leaf up at the
    injection (see 5.3). Seeding in setup() would be too late: setup_acs
    builds the residuals BEFORE the recipe materializes.
    """

    def setup(self, ctx):
        info = ctx.curr.source_info[BRANCH]
        alive = int(ctx.state.branches[BRANCH].inds[0, 0].sum())
        print(f"[sinusoid] leaves alive at move build: {alive}/walker")
        _, moves = SinusoidMoveBuilder(
            wave_gen=get_sinusoid_wave(ctx.curr.general_info),
            num_repeats=info.num_prop_repeats,
            inner_moves=[(StretchMove(), 1.0)],
            move_name="sinusoid_pe",
        ).build(None, ctx.curr, ctx.acs, ctx.priors, ctx.state)
        return moves[0]


print("defined: SinusoidMoveBuilder + SinusoidMove")


defined: SinusoidMoveBuilder + SinusoidMove


### 5.3 Wiring it in — and why the *order* matters

`add_branch` + a `setup_classes` entry + one `Stage` is the whole
registration. Two subtleties decide whether the log-like comes out right:

1. **The leaf must be alive before the residuals are built.** `setup_acs`
   builds each walker's residual from the state's `coords`/`inds`, and it runs
   *before* the recipe's move factories — so seeding a leaf inside a factory is
   too late. The engine seeds a branch it does not know by name from
   **metadata**: a fixed-leaf branch (`nleaves_min == nleaves_max`) comes up
   alive, and a branch declaring an `injection` starts there, scattered by
   `SINUSOID_START_FACTOR` — the same convention stock exposes as
   `MBH_START_FACTOR` / `EMRI_START_FACTOR` / `SOBBH_START_FACTOR`. `0.0` starts
   exactly at truth.
2. **We inject our own data, through our own `signal_gen`.** The stock
   synthetic GB processor injects with GBGPU's fast path, while our template
   comes from `GBTDIonTheFly`; the two agree only to ~1e-8, which leaves a
   small but real residual at truth. Injecting through the very generator the
   engine subtracts with makes the example *exactly* self-consistent — so at
   `SINUSOID_START_FACTOR=0.0` the log-like must come out **~0**.

We also pop the stock GB stages: they would spend every iteration on the GB RJ
search and the recipe would never reach ours.

In [14]:
from eryn.priors import ProbDistContainer, UniformDistribution

# Read at run time by the engine, so setting it here works. (Most env knobs
# resolve as dataclass field defaults at IMPORT, so setting those this late
# would silently do nothing -- num_iterations stays at the lite preset of 3.)
os.environ["SINUSOID_START_FACTOR"] = "0.0"   # start AT truth -> log-like must be ~0

# Fresh backend: a stale one is RESUMED and dies on a shape mismatch. STOCK_DIR
# was already wiped and wired up at the top of the notebook; this run is the
# only one here, so we just make re-running this cell safe.
shutil.rmtree(STOCK_DIR, ignore_errors=True)
_g = erebor.gb_no_fg_lite.general
_g.data_mode, _g.synthetic_injections = "synthetic", "prior"
_g.nwalkers = _g.ntemps = 2
_g.tobs_target = 3 * 86400.0                  # TD generation is the cost: keep the grid short

fit5 = erebor.gb_no_fg_lite()
fit5.resolve_data_source()        # the prior draws happen here (params are None until now)
truth9 = np.asarray(fit5.general.gb_injection_params, dtype=float)[:1]   # ONE source
fit5.general.gb_injection_params = truth9     # keep 1 row -- an EMPTY table crashes the
                                              # stock GB processor (num_sources=0)
print("target sinusoid f0 = %.4f mHz" % (truth9[0, 1] * 1e3))

tr = make_gb_transform_container()
truth_samp = np.asarray(tr.both_inverse_transforms(truth9))     # -> SAMPLING basis (1, 8)

A = truth9[0, 0]
lo, hi = truth9[0, 1] * 0.999, truth9[0, 1] * 1.001
priors = {BRANCH: ProbDistContainer({
    0: UniformDistribution(np.log(A * 0.1), np.log(A * 10.0)),   # ln A
    1: UniformDistribution(lo * 1e3, hi * 1e3),                  # f0 [mHz]
    2: UniformDistribution(-1e-14, 1e-14),                       # fdot
    3: UniformDistribution(0.0, 2 * np.pi),                      # phi0
    4: UniformDistribution(-1.0, 1.0),                           # cos_iota
    5: UniformDistribution(0.0, np.pi),                          # psi
    6: UniformDistribution(0.0, 2 * np.pi),                      # lam
    7: UniformDistribution(-1.0, 1.0),                           # sin_beta
})}

fit5.add_branch(BRANCH, SinusoidSettings(
    ndim=8, nleaves_max=1, nleaves_min=1,     # fixed-leaf -> the engine brings the leaf up alive
    transform=tr, priors=priors,
    periodic={BRANCH: {3: 2 * np.pi, 5: np.pi, 6: 2 * np.pi}},
    branch_state=SinusoidState, branch_backend=None,
    injection=truth_samp,                     # SAMPLING basis -> the engine seeds coords here
    num_prop_repeats=1, waveform_kwargs={},
))
fit5.setup_classes = {**fit5.setup_classes, BRANCH: SinusoidSetup}
fit5.recipe.add_stage(
    Stage("sinusoid_pe", kind="pe",
          moves=[SinusoidMove("sinusoid_pe", branch=BRANCH)]),
    after="gb_pe",
)
for _st in [s.name for s in fit5.recipe.stages]:
    if _st != "sinusoid_pe":                  # this example is about OUR branch only
        fit5.recipe.pop_stage(_st)

print("branches:", list(fit5.branches), "| stages:", [s.name for s in fit5.recipe.stages])

target sinusoid f0 = 7.3835 mHz
branches: ['gb', 'sinusoid'] | stages: ['sinusoid_pe']


In [15]:
curr5 = fit5.build()

# The engine seam, attached post-build (it needs the runtime general_info).
gi5 = curr5.general_info
sig_gen = SinusoidSignalGen(tr, gi5)
curr5.source_info[BRANCH].signal_gen = sig_gen

# Inject OUR signal through the SAME generator the engine subtracts with.
sig = sig_gen(*truth_samp[0])
data5 = gi5.input_data_residual_array
print("data as built (stock GB processor): sum|d| = %.3e"
      % float(np.abs(np.asarray(data5.arr)).sum()))
data5.arr[:] = np.asarray(sig.arr)      # empty sky + only our sinusoid
print("data after OUR injection         : sum|d| = %.3e"
      % float(np.abs(np.asarray(data5.arr)).sum()))

fit5.run()
curr5.summarize_run("sinusoid")

2026-07-16 23:25:14,576 - GeneralSetup - DEBUG - Saving h5 backend to ./gf_output_stock/gb_no_fg_test_2_testing.h5


2026-07-16 23:25:14,576 - GeneralSetup - DEBUG - Saving artifacts to ./gf_output_stock/gb_no_fg_test_2_artifacts/


2026-07-16 23:25:18,345 - GeneralSetup - INFO - Using fixed PSD kwargs: {'psd_params': [1.5e-11, 3e-15], 'galfor_params': None}


2026-07-16 23:25:18,346 - GeneralSetup - DEBUG - Preprocess setting: plot_folder = ./gf_output_stock/gb_no_fg_test_2_artifacts/


2026-07-16 23:25:18,347 - GeneralSetup - DEBUG - Preprocess setting: highpass_kwargs = None


...

2026-07-16 23:25:46,704 - lisatools.globalfit.moves.addremovemove - DEBUG - mean accepted fraction: 0.3333333333333333. elapsed: 2.525052785873413


100%|██████████| 3/3 [00:08<00:00,  2.65s/it]

100%|██████████| 3/3 [00:08<00:00,  2.69s/it]

2026-07-16 23:25:46,752 - lisatools.globalfit.run - INFO - Residuals saved.


=== sinusoid (sampled) ===
branches   : ['gb', 'sinusoid']
log_like   : (3, 2, 2)  final chain (temp 0): [-0. -0.]
  gb      chain (3, 2, 2, 100, 8)          alive-leaves(temp 0,last)=0
  sinusoid chain (3, 2, 2, 1, 8)            alive-leaves(temp 0,last)=2


### Going deeper: what you actually had to write

The log-like above starts at **~0** — the residual at truth is empty, because
our source class injected its own data and the engine subtracted its own
template. That round trip is the check worth copying: set the start factor to
`0.0` and a correct source class *must* return ~0. Raise it (try `1e-3`) and
the likelihood drops away immediately.

Nothing here touched the `gb` branch, and no stock code knows the word
"sinusoid". What made it a first-class citizen:

- **`nleaves_min == nleaves_max`** told the engine it is a fixed-leaf branch, so
  its leaf comes up alive — the same metadata that carries MBH/EMRI/SOBBH. A
  reversible-jump branch (like `gb`, with `nleaves_min = 0`) instead starts
  empty and lets its search move create leaves.
- **`injection` + `SINUSOID_START_FACTOR`** are the stock starting-point
  convention, not something bespoke for this notebook.
- **`signal_gen`** is the only reason the shared residual knows our source
  exists at all.
- **`branch_state=SinusoidState`** is what let us use the add/remove move: it
  reads a per-leaf `betas_all` ladder. `branch_backend=None` is fine — the HDF
  backend skips a `None` sub-backend, so `betas_all` simply is not persisted
  across a resume.

The move itself was `SingleSourcePEBuilder` with one line changed
(`branch_name`). That is the same shared core MBH, EMRI and SOBBH each use, so
a new source class inherits the whole tempered add/remove/PE choreography for
free.

## 6. Instrumentation

**TL;DR.** Debug-capable moves carry a uniform `set_debug` hook; the
recipe exposes it as `fit.set_move_debug(name, ...)` and
`fit.set_stage_debug(stage, ...)`, which flip on a per-move
**residual-trace flip-book** at materialize time. The run persists to the
eryn HDF5 backend (§`01`) and the `postprocessing` /
`diagnosticplot` modules read it back.

In [16]:
demo6 = erebor.gb_no_fg_lite()

# arm one move's debug instrumentation (writes per-leaf residual-trace PNGs)
demo6.set_move_debug("rj_prior", plot_dir="./gb_debug", every=5)
print("rj_prior debug spec:", demo6.recipe.get_move("rj_prior").debug)

# ...or a whole stage at once (each move that has no per-move override)
demo6.set_stage_debug("gb_pe", plot_walker=0)
print("gb_pe stage debug spec:", demo6.recipe._stage("gb_pe").debug)

rj_prior debug spec: {'enabled': True, 'plot_dir': './gb_debug', 'every': 5}
gb_pe stage debug spec: {'enabled': True, 'plot_walker': 0}


### Going deeper: the flip-book and the readout path

- **`set_move_debug` / `set_stage_debug`** stash a `debug` payload on the
  `Move` / `Stage`; `Stage.setup` applies it at materialization via the move's
  `GlobalFitMove.set_debug(enabled, plot_dir=, plot_walker=, plot_leaf=,
  plot_band=, every=)`. The GB special-stretch move and
  `ResidualAddOneRemoveOneMove` then emit a per-step
  `[template | data | residual]` flip-book so you can watch a source
  leave and re-enter the residual. Precedence: per-move > stage
  > the move's `{BRANCH}_DEBUG` env default. Moves without a debug hook
  (e.g. `PSDMove`) carry the flag inertly.
- **Readout:** the run writes a `GFHDFBackend` HDF5 file
  (`curr.backend`, reopenable with `GFHDFBackend(path)`) plus the
  `…_artifacts/` folder (run log, `dump_settings` summary, diagnostic
  plots). `lisatools.globalfit.postprocessing` builds catalogues from the
  chains; `lisatools.globalfit.diagnosticplot` draws the trace/corner/
  log-likelihood figures (see [`01` §Diagnostic plots](01_GlobalFitQuickstart.ipynb)).

## Where to go next

- [`03`](03_StockGlobalFitGallery.ipynb) — a deep, run-it demo of every
  stock variant, branch by branch.
- [`07`](07_ErynSmallToLarge.ipynb) — the eryn sampler the recipe drives
  (walkers, tempering, reversible jump).
- `LISAanalysistools/docs/global-fit-launch.md` (running on ranks/GPUs)
  and `docs/stock-stages-and-moves.md` (the recipe layer).